## Problem 1: Descriptive analysis

In [13]:
import numpy as np
import pandas as pd

# Load data
df = pd.read_stata("A1_kommune.dta")

# Create summary table
variables = ["taxrev", "taxrate", "pop"]
table1 = df[variables].describe(percentiles=[0.25, 0.50, 0.75]).T
table1 = table1[["mean", "std", "min", "50%", "max"]]
table1.columns = ["Mean", "Std. Dev.", "Min", "Median", "Max"]

# Render table: 2 decimal places with no commas, and 0 decimals for pop
table1.style.format("{:.2f}").format(
    subset=pd.IndexSlice[["pop"], :],
    formatter="{:.0f}",
)

,Mean,Std. Dev.,Min,Median,Max
taxrev,4477.34,5251.18,211.23,3317.85,44170.34
taxrate,25.21,0.91,22.80,25.30,27.80
pop,56476,62925,1969,43475,528208


In [14]:
records = []
variables = ["taxrev", "taxrate", "pop"]

for var in variables:
    min_row = df.loc[df[var].idxmin()]
    max_row = df.loc[df[var].idxmax()]

    # Format values based on variable type
    if var == "taxrate":
        min_val = f"{min_row[var]:.2f}%"
        max_val = f"{max_row[var]:.2f}%"
    elif var == "pop":
        min_val = f"{min_row[var]:.0f}"
        max_val = f"{max_row[var]:.0f}"
    else:
        min_val = f"{min_row[var]:.2f}"
        max_val = f"{max_row[var]:.2f}"

    records.append(
        {
            "Variable": var,
            "Lowest": f"{min_row['kommune']} ({min_val})",
            "Highest": f"{max_row['kommune']} ({max_val})",
        }
    )

table_extremes = pd.DataFrame(records).set_index("Variable")
table_extremes.index.name = None
table_extremes

,Lowest,Highest
taxrev,Læsø Kommune (211.23),Københavns Kommune (44170.34)
taxrate,Gentofte Kommune (22.80%),Langeland Kommune (27.80%)
pop,Læsø Kommune (1969),Københavns Kommune (528208)


## Problem 2: Empirical analysis of tax revenues and municipal tax rates

### 2.3

In [15]:
import numpy as np
import statsmodels.formula.api as smf

model1 = smf.ols(formula="np.log(taxrev) ~ taxrate", data=df).fit()

results1 = pd.DataFrame(
    {
        "Parameter estimate": [model1.params["Intercept"], model1.params["taxrate"]],
        "Std. Error": [model1.bse["Intercept"], model1.bse["taxrate"]],
    },
    index=["δ_0", "δ_1"]
)
results1.round(3)

,Parameter estimate,Std. Error
δ_0,11.698,2.143
δ_1,-0.143,0.085
